# Semana 10: análise do aumento em tempo de teste (TTA)

Segunda rodada do script de análise congelado (`analysis/analise_congelada.py`, sem nenhuma mudança), agora com as margens do TTA. É a análise secundária da seção 9 de `prereg/preregistro.md`, rodada depois da abertura da chave dos regimes, com o rótulo **"secundária preregistrada, rodada após abertura da chave"** (`prereg/desvios.md`, entradas de 25/10/2026 e 05/11/2026, com o acompanhamento de 07/11/2026).

O script lê, para cada execução, a margem gravada na semana 6 e a margem do TTA da semana 10. As duas são juntadas em arquivos no disco local; **os arquivos do Drive não são alterados**. Para que uma mudança de biblioteca não altere as tabelas, o script roda com o numpy e o scipy da primeira rodada, instalados à parte.

**O que tem de valer:** as sete tabelas têm de sair idênticas, byte a byte, às da rodada confirmatória. No `resultados.json`, só podem diferir `meta.sem_tta`, `meta.versoes.python` e os valores novos `variantes.<regime>.tta`. A célula 6 confere isso.

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive e confere as pastas, a RAM (mínimo de 20 GB) e o disco |
| 2 | Clona o repositório, confere os SHA-256 dos quatro arquivos do script com `analysis/README.md` e o manifesto do TTA com o registrado no repositório |
| 3 | Instala à parte o numpy e o scipy da primeira rodada |
| 4 | Junta, no disco local, a margem da semana 6 e a margem do TTA de cada execução, conferindo tudo pelo SHA-256, e confere as entradas sem calcular métrica |
| 5 | Roda o script, sem `--sem-tta` (cerca de 3 horas) |
| 6 | Copia a saída para `resultados/semana10/tta_analise` e compara com a rodada confirmatória |

Abra o Colab logada na conta do mestrado, **sem GPU e com Alta RAM** (Ambiente de execução > Alterar tipo de ambiente de execução). **Deixe a aba aberta durante a célula 5.** Se a sessão cair antes da célula 5 terminar, rode de novo desde a célula 1; a célula 4 pula o que já foi montado. **Se alguma célula der erro, pare, não mude nada no script e guarde a saída inteira.**

Depois da célula 6, me mande `rodada.json`, `tta_resumo.csv` e `analise.log`, da pasta `resultados/semana10/tta_analise`.

In [ ]:
# Célula 1 de 6: Drive, pastas, memória e disco
import os, re, json, time, hashlib, shutil, subprocess, sys, csv, math
import numpy as np
import psutil

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

MARGENS6 = f'{RAIZ}/resultados/semana6/margens'
TTA = f'{RAIZ}/resultados/semana10/tta_margens'
CHAVE = f'{RAIZ}/resultados/semana6_chave/chave_codigos.json'
PRIMEIRA = f'{RAIZ}/resultados/semana6/analise'
SAIDA_DRIVE = f'{RAIZ}/resultados/semana10/tta_analise'
LOCAL = '/content/margens_tta'          # só os 64 arquivos que o script lê
TMP = '/content/tmp_tta'
MARCAS = '/content/marcas_tta'
assert os.path.isfile(f'{MARGENS6}/manifesto_sha256.csv'), 'Manifesto das margens da semana 6 não encontrado.'
assert os.path.isfile(f'{TTA}/manifesto_sha256.csv'), 'O notebook semana10_tta_margens não terminou: falta o manifesto em resultados/semana10/tta_margens.'
assert os.path.isfile(f'{PRIMEIRA}/resultados.json'), 'Saída da rodada confirmatória não encontrada em resultados/semana6/analise.'
assert os.path.isfile(CHAVE), 'A chave dos códigos não foi encontrada.'
assert not os.path.exists(f'{SAIDA_DRIVE}/resultados.json'), (
    'Já existe uma rodada em resultados/semana10/tta_analise. Nada é gravado por cima dela: '
    'para outra rodada, mude SAIDA_DRIVE para resultados/semana10/tta_analise_2.')
for d in (LOCAL, TMP, MARCAS):
    os.makedirs(d, exist_ok=True)

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

ram = psutil.virtual_memory().total / 1e9
disco = shutil.disk_usage('/content')[2] / 1e9
print(f'RAM da sessão: {ram:.1f} GB. Disco local livre: {disco:.0f} GB.')
assert ram >= 20, ('RAM abaixo de 20 GB. Com o TTA, a análise usa mais memória que a rodada confirmatória. '
                   'Escolha Alta RAM em Ambiente de execução > Alterar tipo de ambiente de execução.')
assert disco > 40, 'Disco local com menos de 40 GB livres (as margens juntadas ocupam cerca de 18 GB).'

In [ ]:
# Célula 2 de 6: repositório, script congelado e manifesto do TTA
REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO = '/content/repo'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()

# os quatro arquivos do script têm de ser os congelados em analysis/README.md
tabela = open(f'{REPO}/analysis/README.md', encoding='utf-8').read()
registrados = dict(re.findall(r'\| `([a-z_]+\.py)` \| `([0-9a-f]{64})` \|', tabela))
assert set(registrados) == {'analise_congelada.py', 'nucleo.py', 'reamostragem.py', 'categorias.py'}, registrados.keys()
for nome, sha in registrados.items():
    assert sha256_arquivo(f'{REPO}/analysis/{nome}') == sha, f'{nome} é diferente do congelado. Não rode a análise.'

# o manifesto das margens do TTA no Drive tem de ser o registrado no repositório
MAN_TTA = list(csv.DictReader(open(f'{TTA}/manifesto_sha256.csv', newline='')))
assert MAN_TTA == list(csv.DictReader(open(f'{REPO}/analysis/tta/margens/manifesto_sha256.csv', newline=''))), (
    'O manifesto do TTA no Drive é diferente do registrado em analysis/tta/margens. Pare e guarde esta saída.')
assert len(MAN_TTA) == 45, f'O manifesto do TTA tem {len(MAN_TTA)} arquivos, esperado 45.'
print(f'Repositório no commit {commit[:7]}. Os quatro arquivos do script conferem com analysis/README.md, '
      'e o manifesto do TTA confere com o do repositório.')

In [ ]:
# Célula 3 de 6: numpy e scipy da primeira rodada, instalados à parte
PINADO = '/content/pinado'
META1 = json.load(open(f'{PRIMEIRA}/resultados.json'))['meta']
V1 = META1['versoes']
print(f'Versões da rodada confirmatória: Python {V1["python"]}, numpy {V1["numpy"]}, scipy {V1["scipy"]}.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-deps', '--target', PINADO,
                f'numpy=={V1["numpy"]}', f'scipy=={V1["scipy"]}'], check=True)
AMBIENTE = dict(os.environ, PYTHONPATH=PINADO)
r = subprocess.run([sys.executable, '-c', 'import sys, numpy, scipy; print(sys.version.split()[0], numpy.__version__, scipy.__version__, numpy.__file__)'],
                   env=AMBIENTE, capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
py, vnp, vsp, onde = r.stdout.split()
assert vnp == V1['numpy'] and vsp == V1['scipy'] and onde.startswith(PINADO), (
    f'O script veria numpy {vnp} e scipy {vsp} ({onde}), não os da primeira rodada. Pare e guarde esta saída.')
print(f'O script vai rodar com numpy {vnp} e scipy {vsp} de {PINADO}.')
if py != V1['python']:
    print(f'Python {py} (a primeira rodada usou {V1["python"]}); a diferença está registrada em prereg/desvios.md.')

In [ ]:
# Célula 4 de 6: margens juntadas no disco local e conferência das entradas, sem calcular métrica
MAN6 = list(csv.DictReader(open(f'{MARGENS6}/manifesto_sha256.csv', newline='')))
assert len(MAN6) == 64, f'O manifesto da semana 6 tem {len(MAN6)} arquivos, esperado 64 (60 execuções e 4 rótulos).'
SHA_TTA = {l['arquivo']: l['sha256'] for l in MAN_TTA}
COM_TTA = ('teste', 'bolivia', 't2')
RX = re.compile(r'run_([0-9a-z]+)_s(\d)_([a-z0-9]+)\.npz')

def copiar_conferido(origem, pasta, sha):
    destino = f'{pasta}/{os.path.basename(origem)}'
    if not (os.path.exists(destino) and sha256_arquivo(destino) == sha):
        parcial = f'{pasta}/parcial_{os.path.basename(origem)}'
        shutil.copyfile(origem, parcial)
        os.replace(parcial, destino)
    assert sha256_arquivo(destino) == sha, f'{os.path.basename(origem)} diferente do manifesto. Pare e guarde esta saída.'
    return destino

def sha_vetor(v):
    return hashlib.sha256(np.ascontiguousarray(v).tobytes()).hexdigest()

ENTRADA = []
t0 = time.time()
for l in MAN6:
    nome = l['arquivo']
    m = RX.fullmatch(nome)
    if m is None or m.group(3) not in COM_TTA:          # rótulos e validação: cópia idêntica
        copiar_conferido(f'{MARGENS6}/{nome}', LOCAL, l['sha256'])
        ENTRADA.append([nome, 'semana 6, sem mudança', l['sha256']])
        continue
    cod, sem, conj = m.groups()
    nome_tta = f'tta_{cod}_s{sem}_{conj}.npz'
    marca = f'{MARCAS}/{nome}.json'
    if os.path.exists(marca) and os.path.exists(f'{LOCAL}/{nome}'):
        linha = json.load(open(marca))
        assert sha256_arquivo(f'{LOCAL}/{nome}') == linha[2], f'{nome} local mudou depois de montado.'
        ENTRADA.append(linha)
        continue
    a6 = copiar_conferido(f'{MARGENS6}/{nome}', TMP, l['sha256'])
    at = copiar_conferido(f'{TTA}/{nome_tta}', TMP, SHA_TTA[nome_tta])
    with np.load(a6, allow_pickle=False) as z6, np.load(at, allow_pickle=False) as zt:
        assert set(z6.files) == {'chaves', 'tamanhos', 'margem'} and set(zt.files) == {'chaves', 'tamanhos', 'margem_tta'}, nome
        assert np.array_equal(z6['chaves'], zt['chaves']) and np.array_equal(z6['tamanhos'], zt['tamanhos']), f'{nome}: unidades diferentes.'
        margem, tta = z6['margem'], zt['margem_tta']
        assert margem.shape == tta.shape and tta.dtype == np.float32 and np.isfinite(tta).all(), nome
        sha_m, sha_t = sha_vetor(margem), sha_vetor(tta)
        parcial = f'{LOCAL}/parcial_{nome}'
        np.savez(parcial, chaves=z6['chaves'], tamanhos=z6['tamanhos'], margem=margem, margem_tta=tta)
    with np.load(parcial, allow_pickle=False) as z:
        assert sha_vetor(z['margem']) == sha_m and sha_vetor(z['margem_tta']) == sha_t, f'{nome}: arquivo juntado não confere.'
    os.replace(parcial, f'{LOCAL}/{nome}')
    linha = [nome, f'margem de {nome} (semana 6) e margem_tta de {nome_tta} (semana 10)', sha256_arquivo(f'{LOCAL}/{nome}')]
    json.dump(linha, open(marca, 'w'))
    ENTRADA.append(linha)
    os.remove(a6); os.remove(at)
    juntados = sum(1 for e in ENTRADA if 'margem_tta' in e[1])
    if juntados % 9 == 0:
        print(f'   {juntados} de 45 arquivos juntados ({(time.time() - t0) / 60:.0f} min)')
assert sorted(os.listdir(LOCAL)) == sorted(l['arquivo'] for l in MAN6), 'A pasta local tem arquivos além dos 64.'
assert sum(1 for e in ENTRADA if 'margem_tta' in e[1]) == 45
print(f'64 arquivos prontos em {LOCAL} ({(time.time() - t0) / 60:.0f} min): 45 com margem e margem_tta, 19 sem mudança.')

verificar = r'''
import sys
sys.path.insert(0, sys.argv[1] + "/analysis")
import numpy as np
import analise_congelada as A
ctx = A.montar_contexto(sys.argv[1], sys.argv[2])
corridas = A.listar_corridas(sys.argv[2])
codigos = A.verificar_corridas(corridas)
for cod in corridas:
    for sem in corridas[cod]:
        for conj, caminho in corridas[cod][sem].items():
            with np.load(caminho) as z:
                assert ("margem_tta" in z.files) == (conj != "val"), (cod, sem, conj)
print("Códigos:", " ".join(sorted(codigos)), "| sementes por código:", [len(corridas[c]) for c in sorted(codigos)])
print("Unidades: fonte", ctx["fonte"].n, "| alvo", ctx["alvo"].n)
'''
r = subprocess.run([sys.executable, '-c', verificar, REPO, LOCAL], env=AMBIENTE, capture_output=True, text=True)
print(r.stdout.strip())
assert r.returncode == 0, r.stderr[-3000:]
print('Entradas conferidas. Nenhuma métrica foi calculada.')

In [ ]:
# Célula 5 de 6: segunda rodada do script congelado, com o TTA (cerca de 3 horas). Deixe esta aba aberta.
SAIDA = '/content/analise_tta'
LOG = '/content/analise_tta.log'
assert not os.path.exists(f'{SAIDA}/resultados.json'), (
    'Esta sessão já tem uma saída completa em /content/analise_tta. Não rode o script de novo: vá para a célula 6.')
cmd = (f'cd {REPO} && set -o pipefail && PYTHONPATH={PINADO} {sys.executable} analysis/analise_congelada.py --repo . '
       f'--margens {LOCAL} --chave "{CHAVE}" --saida {SAIDA} 2>&1 | tee {LOG}')
get_ipython().system(f"bash -c '{cmd}'")

In [ ]:
# Célula 6 de 6: cópia para o Drive e comparação com a rodada confirmatória
assert os.path.isfile(f'{SAIDA}/resultados.json'), 'A análise não terminou: não há resultados.json. Guarde o log e me mande.'
os.makedirs(SAIDA_DRIVE, exist_ok=True)
shutil.copytree(SAIDA, SAIDA_DRIVE, dirs_exist_ok=True)
shutil.copy(LOG, f'{SAIDA_DRIVE}/analise.log')

TABELAS = ('principal', 'sensibilidades', 'eventos', 'regimes', 'precondicoes', 'execucoes', 'rq3_por_unidade')
IGUAIS = {t: sha256_arquivo(f'{PRIMEIRA}/tabelas/{t}.csv') == sha256_arquivo(f'{SAIDA}/tabelas/{t}.csv') for t in TABELAS}

def iguais(a, b):
    return a == b or (isinstance(a, float) and isinstance(b, float) and math.isnan(a) and math.isnan(b))

def diferencas(x, y, caminho=''):
    if isinstance(x, dict) and isinstance(y, dict):
        saida = []
        for k in sorted(set(x) | set(y)):
            if k not in x:
                saida.append(('só na segunda', f'{caminho}/{k}'))
            elif k not in y:
                saida.append(('só na primeira', f'{caminho}/{k}'))
            else:
                saida += diferencas(x[k], y[k], f'{caminho}/{k}')
        return saida
    if isinstance(x, list) and isinstance(y, list) and len(x) == len(y):
        return [d for i, (a, b) in enumerate(zip(x, y)) for d in diferencas(a, b, f'{caminho}/{i}')]
    return [] if iguais(x, y) else [('valor diferente', caminho)]

def permitida(caminho):
    return caminho in ('/meta/sem_tta', '/meta/versoes/python') or re.fullmatch(r'/variantes/R[123]/tta', caminho) is not None

R1 = json.load(open(f'{PRIMEIRA}/resultados.json'))
R2 = json.load(open(f'{SAIDA}/resultados.json'))
DIFS = diferencas(R1, R2)
FORA = [d for d in DIFS if not permitida(d[1])]
IDENTICAS = all(IGUAIS.values()) and not FORA and all(f'/variantes/{r}/tta' in [d[1] for d in DIFS] for r in ('R1', 'R2', 'R3'))

# resumo do TTA, lido do resultados.json da segunda rodada (nenhum cálculo novo)
RESUMO = []
for r in ('R1', 'R2', 'R3'):
    v = R2['variantes'][r]
    for medida in ('auroc_e_alvo', 'auroc_e_fonte', 'augrc_alvo'):
        for variante in ('unico', 'tta'):
            e = v[variante][medida]
            RESUMO.append([r, medida, variante, e['ponto'], e['ic90'][0], e['ic90'][1], ''])
    for variante in ('unico', 'tta'):
        RESUMO.append([r, 'eaurc_alvo', variante, v[variante]['eaurc_alvo'], '', '', ''])
    e = v['tta']['diferenca_auroc_e_alvo_menos_unico']
    RESUMO.append([r, 'auroc_e_alvo_tta_menos_unico', 'diferenca', e['ponto'], e['ic90'][0], e['ic90'][1], e['p']])
with open(f'{SAIDA_DRIVE}/tta_resumo.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['regime', 'medida', 'variante', 'ponto', 'ic90_inferior', 'ic90_superior', 'p'])
    w.writerows(RESUMO)
with open(f'{SAIDA_DRIVE}/manifesto_entrada_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['arquivo', 'origem', 'sha256'])
    w.writerows(ENTRADA)

ROTULO = 'secundaria preregistrada, rodada apos abertura da chave'
INFO = {'rotulo': ROTULO, 'commit_repositorio': commit, 'scripts_sha256': registrados,
        'comando': 'analise_congelada.py (sem --sem-tta), margens juntadas da semana 6 e da semana 10',
        'versoes': R2['meta']['versoes'], 'versoes_primeira_rodada': V1,
        'tabelas_identicas_a_primeira_rodada': IGUAIS, 'diferencas_no_resultados_json': [list(d) for d in DIFS],
        'diferencas_nao_permitidas': [list(d) for d in FORA],
        'conclusao': ('tabelas primarias identicas; valores do TTA reportaveis como ' + ROTULO) if IDENTICAS
                     else 'tabelas primarias NAO identicas; a primeira rodada continua sendo o registro e o TTA nao e reportado',
        'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_DRIVE}/rodada.json', 'w'), indent=2, ensure_ascii=False)

for t, ok in IGUAIS.items():
    print(f'{t}.csv: {"idêntica" if ok else "DIFERENTE"} à da rodada confirmatória')
print('Diferenças no resultados.json:', [d[1] for d in DIFS])
if IDENTICAS:
    print('\nAs tabelas primárias saíram idênticas. Resumo do TTA (média dos 5 seeds, IC de 90%):')
    for linha in RESUMO:
        r, medida, variante, ponto, li, ls, p = linha
        ic = f' [{li:.4f}; {ls:.4f}]' if li != '' else ''
        print(f'   {r} {medida:30s} {variante:9s} {ponto:.4f}{ic}' + (f'  p = {p:.3f}' if p != '' else ''))
else:
    print('\nATENÇÃO: as tabelas primárias NÃO saíram idênticas. Não use os valores do TTA. Me mande rodada.json e analise.log.')
print(f'\nSaída em {SAIDA_DRIVE.replace(MEUDRIVE, "MyDrive")}: rodada.json, tta_resumo.csv, analise.log, tabelas e resultados.json.')